# 5.4 · 朴素贝叶斯 / Naive Bayes

> **课程定位 / Where this fits**
> 第 4 课，**Part 5 · 监督学习：分类**。
> Lesson 4, **Part 5 · Supervised Classification**.
>
> 前几课直接画决策边界（判别式）。朴素贝叶斯是**生成式**模型的代表：用贝叶斯定理(2.8)对"给定特征，属于某类的概率"建模，再"朴素"地假设特征之间**条件独立**。简单、极快，在文本分类（垃圾邮件）上出奇地好。
> Earlier lessons drew decision boundaries directly (discriminative). Naive Bayes is the canonical **generative** model: it uses Bayes' theorem (2.8) to model "probability of a class given the features", with the "naive" assumption that features are **conditionally independent**. Simple, very fast, and surprisingly strong on text (spam).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $c$ —— 类别 / a class
> - $x_j$ —— 第 $j$ 个特征 / the $j$-th feature
> - $\Pr(c)$ —— 类先验 / class prior
> - $\Pr(x_j\mid c)$ —— 似然（某类下该特征的分布）/ likelihood (feature distribution within a class)
> - $\alpha$ —— 拉普拉斯平滑系数 / Laplace smoothing constant

> 💡 **面试相关 / Interview-relevant**
> - "朴素贝叶斯的'朴素'假设是什么 / 为什么仍有效"（★★★★★）
> - "Gaussian / Multinomial / Bernoulli NB 的区别"（★★★★）
> - "拉普拉斯平滑解决什么问题"（★★★★，零概率）
> - "为什么在对数空间计算"（★★★★，下溢）
> - "生成式 vs 判别式"（★★★★）

---

## 学习目标 / Learning Objectives

1. 用贝叶斯定理推出朴素贝叶斯，看清"朴素"假设。
   Derive Naive Bayes from Bayes' theorem and see the "naive" assumption.
2. 区分三种变体：Gaussian / Multinomial / Bernoulli。
   Distinguish the three variants: Gaussian / Multinomial / Bernoulli.
3. 理解**拉普拉斯平滑**防零概率、**对数空间**防下溢。
   Understand Laplace smoothing (avoids zero probability) and log-space (avoids underflow).
4. **从零**实现并做文本垃圾短信分类。
   Implement **from scratch** and classify spam SMS.
5. 说清生成式 vs 判别式。
   Articulate generative vs discriminative.

## 目录 / TOC
1. [先建直觉：用贝叶斯反推类别](#1)
2. [贝叶斯定理到 NB ⭐](#2)
3. [三种变体 ⭐](#3)
4. [📧 数据：SMS 垃圾短信](#4)
5. [从零实现 + 平滑 + 对数 ⭐](#5)
6. [对照 sklearn + 高斯 NB](#6)
7. [生成式 vs 判别式 ⭐](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：用贝叶斯反推类别 / Intuition First

判断一封邮件是不是垃圾，我们其实在问：**"看到这些词，它是垃圾邮件的概率有多大？"** 直接估这个很难，但反过来容易——**"垃圾邮件里出现这些词的概率"** 我们可以数出来。贝叶斯定理(2.8)正是把"反过来"翻回"正着问"的桥梁。
To judge if an email is spam, we're really asking: **"given these words, how likely is it spam?"** Estimating that directly is hard, but the reverse is easy — **"how likely are these words in spam"** can just be counted. Bayes' theorem (2.8) is the bridge that flips the easy direction back into the question we want.

"朴素"在哪？为了简化，我们假设**各个词的出现互不影响**（给定类别后条件独立）。这显然不完全对（"免费"和"领取"常一起出现），但**即便假设错了，分类结果往往还是对的**——下面解释为什么。
Where's the "naive"? To simplify, we assume **words appear independently of each other** (conditionally independent given the class). This is clearly not fully true ("free" and "claim" co-occur), but **even when the assumption is wrong, the classification is often still right** — explained below.


<a id="2"></a>
## 2. 贝叶斯定理到朴素贝叶斯 ⭐ / Bayes → Naive Bayes

给定特征 $\mathbf{x}=(x_1,\dots,x_d)$，类 $c$ 的后验由贝叶斯定理给出：
Given features $\mathbf{x}=(x_1,\dots,x_d)$, the posterior of class $c$ comes from Bayes' theorem:

$$\Pr(c\mid\mathbf{x}) = \frac{\Pr(c)\,\Pr(\mathbf{x}\mid c)}{\Pr(\mathbf{x})} \;\propto\; \Pr(c)\,\Pr(\mathbf{x}\mid c)$$

（分母 $\Pr(\mathbf{x})$ 对所有类都一样，比较时可忽略，所以用 $\propto$。）
(The denominator $\Pr(\mathbf{x})$ is the same for every class, so we drop it — hence $\propto$.)

难点是似然 $\Pr(\mathbf{x}\mid c)=\Pr(x_1,\dots,x_d\mid c)$——联合分布的维度会爆炸。
The hard part is the likelihood $\Pr(\mathbf{x}\mid c)=\Pr(x_1,\dots,x_d\mid c)$ — the joint distribution explodes in dimension.

**"朴素"假设**：给定类别，各特征**条件独立**，于是联合似然拆成连乘：
The **"naive" assumption**: features are **conditionally independent** given the class, so the joint likelihood factorizes into a product:

$$\Pr(\mathbf{x}\mid c) = \prod_{j=1}^{d} \Pr(x_j\mid c)$$

分类规则就是取后验最大的类（MAP）：
The classification rule is the class with the largest posterior (MAP):

$$\hat{c} = \arg\max_c \;\Pr(c)\prod_j \Pr(x_j\mid c)$$

**为什么"明显错误"的独立假设仍有效**（面试爱问）：即使概率估得不够准，只要**正确类别的后验仍然是最大的**，分类就对。NB 要的是"把正确类排到第一"，不是"概率绝对准确"。
**Why the obviously-wrong independence still works** (a favorite question): even if probabilities are off, as long as the **correct class still has the largest posterior**, the prediction is right. NB only needs the correct class ranked first, not exact probabilities.


<a id="3"></a>
## 3. 三种变体 ⭐ / Three Variants

三种变体的区别**只在于怎么建模 $\Pr(x_j\mid c)$**：
The three variants differ **only in how they model $\Pr(x_j\mid c)$**:

| 变体 variant | 特征类型 feature type | $\Pr(x_j\mid c)$ 模型 | 典型场景 |
|---|---|---|---|
| **Gaussian** | 连续 continuous | 正态 $\mathcal{N}(\mu_{jc},\sigma_{jc}^2)$ | Iris 等数值特征 |
| **Multinomial** | 计数 counts | 多项（词频）/ multinomial (word counts) | 文本（词袋计数）|
| **Bernoulli** | 0/1 | 伯努利（出现与否）/ Bernoulli (present or not) | 文本（词是否出现）|


<a id="4"></a>
## 4. 数据：SMS 垃圾短信 / SMS Spam (mini)

内联一个小型双语垃圾短信集（同 3.7 风格）。任务：判断一条短信是 **spam（垃圾）** 还是 **ham（正常）**。
We inline a small spam-SMS set (same style as 3.7). Task: classify each message as **spam** or **ham** (normal).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

spam = [
    "WIN a FREE prize now click here", "free entry to win cash prize", "claim your free reward urgent",
    "you won a lottery call now", "cheap loans apply free today", "free gift card click link now",
    "urgent win money prize claim", "congratulations you won free cash", "limited offer free click now",
    "win free iphone click here urgent",
]
ham = [
    "are we still meeting for lunch today", "can you send me the report please", "happy birthday see you tonight",
    "the meeting is moved to 3pm", "thanks for your help yesterday", "let me know when you arrive home",
    "i will call you after work today", "please review the document when free", "see you at the gym later",
    "did you finish the homework yet",
]
texts = spam + ham
labels = np.array([1]*len(spam) + [0]*len(ham))   # 标签: 1=spam(垃圾), 0=ham(正常)
print(f"SMS: {len(texts)} 条 ({labels.sum()} spam / {(labels==0).sum()} ham)")
print("spam 例:", spam[0]); print("ham  例:", ham[0])


<a id="5"></a>
## 5. 从零实现 + 平滑 + 对数 ⭐ / From Scratch

先把文本变成**词频矩阵**（`CountVectorizer`：每行一条短信，每列一个词，值是出现次数）。然后实现 Multinomial NB，注意两个工程关键：
First turn text into a **word-count matrix** (`CountVectorizer`: one row per message, one column per word, value = count). Then implement Multinomial NB, minding two engineering essentials:

- **拉普拉斯平滑 / Laplace smoothing**：若某词在某类训练中从未出现，$\Pr(\text{词}\mid c)=0$，会让整个连乘瞬间归零。给每个计数加一个 $\alpha$（通常 1），让未见过的词也有一点点概率：$\frac{\text{count}+\alpha}{\text{total}+\alpha V}$。
  If a word never appeared in a class, $\Pr(\text{word}\mid c)=0$ zeros out the whole product. Add $\alpha$ (usually 1) to each count so unseen words get a small probability.
- **对数空间 / Log-space**：很多小概率连乘会**下溢**到 0（浮点数太小）。取对数把"连乘"变成"连加"，数值稳定：$\log\Pr(c)+\sum_j\log\Pr(x_j\mid c)$。
  Multiplying many tiny probabilities **underflows** to 0. Take logs to turn the product into a sum, which is numerically stable.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

vec = CountVectorizer()
Xc = vec.fit_transform(texts).toarray()   # 转成词频矩阵: 形状 (短信数, 词表大小), 值=该词出现次数
Xtr, Xte, ytr, yte = train_test_split(Xc, labels, test_size=0.3, stratify=labels, random_state=0)

class MultinomialNB_scratch:
    def __init__(self, alpha=1.0): self.alpha = alpha     # alpha = 拉普拉斯平滑强度
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.log_prior = {}; self.log_lik = {}
        for c in self.classes:
            Xc = X[y == c]                                # 取出属于类别 c 的所有短信
            self.log_prior[c] = np.log(len(Xc) / len(X))  # 先验 log P(c) = 该类占比取对数
            counts = Xc.sum(axis=0) + self.alpha          # 该类下每个词的总次数 + 平滑 α(防止0)
            self.log_lik[c] = np.log(counts / counts.sum())  # 似然 log P(词|c), 归一化后取对数
        return self
    def predict(self, X):
        out = []
        for x in X:                                       # 对每条待预测短信 x(它是词频向量)
            # 每个类的得分 = log先验 + Σ(词频 × log似然); 用加法代替连乘(对数空间)
            scores = {c: self.log_prior[c] + (x * self.log_lik[c]).sum()
                      for c in self.classes}
            out.append(max(scores, key=scores.get))       # 取得分最高的类
        return np.array(out)

nb = MultinomialNB_scratch().fit(Xtr, ytr)
print(f"从零 Multinomial NB 准确率 from-scratch accuracy: {(nb.predict(Xte) == yte).mean():.3f}")

# 看哪些词最"像垃圾邮件": 两类 log似然之差越大, 越偏向 spam / most spam-indicative words
log_ratio = nb.log_lik[1] - nb.log_lik[0]                 # log P(词|spam) - log P(词|ham)
top = np.argsort(log_ratio)[-6:][::-1]                    # 取差值最大的 6 个词
words = np.array(vec.get_feature_names_out())
print("最像 spam 的词 most spammy words:", list(words[top]))


<a id="6"></a>
## 6. 对照 sklearn + 高斯 NB / sklearn & Gaussian NB

文本是计数特征，用 `MultinomialNB`；连续特征（如 Iris）用 `GaussianNB`（假设每类每特征服从正态，用训练集的均值/标准差估计）。
Text uses count features → `MultinomialNB`; continuous features (e.g. Iris) → `GaussianNB` (assumes each feature is normal within each class, estimated from the training mean/std).


In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB
sk = MultinomialNB(alpha=1.0).fit(Xtr, ytr)
print(f"sklearn MultinomialNB 准确率 accuracy: {sk.score(Xte, yte):.3f}  (与从零一致 matches)")

# Gaussian NB 处理连续特征(Iris) / Gaussian NB on continuous Iris
from sklearn.datasets import load_iris
iris = load_iris()
ix_tr, ix_te, iy_tr, iy_te = train_test_split(iris.data, iris.target, test_size=0.3,
                                              stratify=iris.target, random_state=0)
gnb = GaussianNB().fit(ix_tr, iy_tr)
print(f"Gaussian NB on Iris 准确率 accuracy: {gnb.score(ix_te, iy_te):.3f}")
print("Gaussian NB: 假设每类每特征服从正态, 用训练集 μ,σ 估计 / assumes per-class normal features")
print("注: NB 不需要缩放(每特征独立建模) — 与 KNN/SVM 不同 / NB needs no scaling")


<a id="7"></a>
## 7. 生成式 vs 判别式 ⭐ / Generative vs Discriminative

| | 生成式 generative (NB, LDA) | 判别式 discriminative (逻辑回归, SVM) |
|---|---|---|
| 建模对象 models | 联合 $\Pr(\mathbf{x}, c)$ → 反推后验 | 直接 $\Pr(c\mid\mathbf{x})$ 或边界 |
| 数据少时 small data | 收敛快、更稳 | 易过拟合 |
| 数据多时 big data | 受错误假设拖累 | 通常更准 |
| 副产品 byproduct | 能**生成**新样本 | 不能 |

**经典结论**（Ng & Jordan）：小数据时 NB 往往更好，大数据时逻辑回归会追上并超过。NB 训练**极快**（只数频次），是高维文本的强基线。
**Classic result** (Ng & Jordan): NB often wins on small data; logistic regression catches up and surpasses on large data. NB trains **very fast** (just counting), a strong baseline for high-dimensional text.


In [ ]:
from sklearn.linear_model import LogisticRegression
print("文本任务上 NB vs 逻辑回归(本小数据) / NB vs logistic on this tiny text set:")
print(f"  MultinomialNB:      {sk.score(Xte, yte):.3f}")
print(f"  LogisticRegression: {LogisticRegression(max_iter=1000).fit(Xtr,ytr).score(Xte,yte):.3f}")
print("生成式(NB)只数频次, 训练 O(数据量); 判别式(LR)要迭代优化 / NB counts, LR iterates")


<a id="8"></a>
## 8. 小结 / Summary

```
朴素贝叶斯: P(c|x) ∝ P(c)∏P(xⱼ|c) (贝叶斯 2.8 + 条件独立"朴素"假设)
即使独立假设错, 只要正确类后验最大 → 分类仍对
变体: Gaussian(连续) / Multinomial(计数,文本) / Bernoulli(0-1)
工程: 拉普拉斯平滑防零概率; 对数空间防下溢
不需缩放; 训练极快(数频次); 高维文本强基线
生成式(建模联合, 能生成) vs 判别式(直接建边界): 小数据 NB 优, 大数据 LR 优
```

### 💡 面试速查 / Interview cheat-sheet
1. **"朴素"= 给定类别后特征条件独立**；假设常错但分类仍有效（只需排序对）。
   "Naive" = conditional independence; often wrong yet still works (only ranking matters).
2. **三变体**按特征类型选：连续→Gaussian，词频→Multinomial，0/1→Bernoulli。
   Pick the variant by feature type.
3. **拉普拉斯平滑**防未见词把连乘归零；**对数空间**防下溢。
   Laplace smoothing avoids zero products; log-space avoids underflow.
4. **生成式 vs 判别式**：NB 训练快、小数据稳；LR 大数据更准。
   Generative vs discriminative: NB fast and stable on small data; LR better on big data.

### 下一节 / Next
**5.5 SVM**——回到判别式。SVM 用"最大间隔"找最稳健的边界，核技巧让它能画非线性边界，曾是深度学习前的王者。
**5.5 SVM** — back to discriminative. SVM finds the maximum-margin boundary, and the kernel trick lets it draw nonlinear ones; the pre-deep-learning champion.
